Environment

This notebook is intended to run in the project virtual environment:

Python (ns-health-signals)

All dependencies are listed in requirements.txt.

# EMS / EHS Demand Signal - Nova Scotia

This notebook processes Nova Scotia Emergency Health Services (EHS) data to create a monthly time series of EMS demand and/or performance metrics by Nova Scotia Health Zone.

## Purpose
EMS activity and performance (responses, response time, offload delay) reflect downstream acute demand and system strain.  
This signal will later be merged with primary care access and hospital indicators to support an early-warning model.

## Output

This notebook produces a processed EMS demand dataset:

`data_processed/ehs_demand_ns.csv`

containing:

- Zone
- Date (monthly)
- ehs_responses
- responses_roll3 (3-month rolling mean)
- responses_pct_change (month-to-month % change)

This dataset will later be merged with primary care access and hospital utilization indicators to support an early-warning model.

## Load EMS Dataset

Load the Nova Scotia Emergency Health Services (EHS) responses dataset from the project raw data folder and inspect its structure.

This dataset contains monthly EMS activity metrics by Nova Scotia Health zone.

In [1]:
import pandas as pd

ehs_path = "data_raw/Emergency_Health_Services_20260301.csv"
ehs = pd.read_csv(ehs_path)

ehs.head()

,Zone,Hospital,Type,Date,Measure Name,Actual,CTAS
0,Eastern,NaN,NaN,2023-12-01,EHS Responses,452,NaN
1,Central,NaN,NaN,2023-12-01,EHS Responses,923,NaN
2,Central,NaN,NaN,2023-12-01,EHS Responses,"3,569",NaN
3,Eastern,NaN,NaN,2023-12-01,EHS Responses,"1,697",NaN
4,Northern,NaN,NaN,2023-12-01,EHS Responses,470,NaN


## Inspect Dataset Structure

Examine column names and available EMS measures to identify the appropriate demand signal.

In [2]:
ehs.columns

Index(['Zone', 'Hospital', 'Type', 'Date', 'Measure Name', 'Actual', 'CTAS'], dtype='object')

## Explore EMS Measures

Identify which EMS indicators represent response volume (demand) rather than performance metrics.

In [3]:
ehs["Measure Name"].value_counts(dropna=False)

Measure Name
EHS Responses    140
Name: count, dtype: int64

## Preview EMS response records

Review sample rows of the selected EMS response measure to confirm structure and values prior to aggregation.

In [4]:
ehs.head(20)

,Zone,Hospital,Type,Date,Measure Name,Actual,CTAS
0,Eastern,NaN,NaN,2023-12-01,EHS Responses,452,NaN
1,Central,NaN,NaN,2023-12-01,EHS Responses,923,NaN
2,Central,NaN,NaN,2023-12-01,EHS Responses,"3,569",NaN
3,Eastern,NaN,NaN,2023-12-01,EHS Responses,"1,697",NaN
4,Northern,NaN,NaN,2023-12-01,EHS Responses,470,NaN
5,Northern,NaN,NaN,2023-12-01,EHS Responses,"1,961",NaN
6,Western,NaN,NaN,2023-12-01,EHS Responses,563,NaN
7,Western,NaN,NaN,2023-12-01,EHS Responses,"2,165",NaN
8,Eastern,NaN,NaN,2023-11-01,EHS Responses,"1,672",NaN
9,Central,NaN,NaN,2023-11-01,EHS Responses,"3,647",NaN


## Define EMS demand signal

EMS demand is defined as the total number of Emergency Health Services responses per month aggregated by Nova Scotia Health zone.

## Prepare EMS demand dataset

For the EMS early-warning signal, we focus on **response volume** (demand).  
We keep only the fields needed to build a monthly demand time series (Zone, Date, Actual) and remove records missing these key values.

In [5]:
ehs_demand = ehs[["Zone", "Date", "Actual"]].copy()
ehs_demand = ehs_demand.dropna(subset=["Zone", "Date", "Actual"])
ehs_demand.head()

,Zone,Date,Actual
0,Eastern,2023-12-01,452
1,Central,2023-12-01,923
2,Central,2023-12-01,"3,569"
3,Eastern,2023-12-01,"1,697"
4,Northern,2023-12-01,470


## Clean numeric and date fields

Convert EMS response counts from text (with thousands separators) to numeric values and parse Date as datetime for time series aggregation.

In [6]:
import pandas as pd

ehs_demand["Actual"] = (
    ehs_demand["Actual"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .astype(float)
)

ehs_demand["Date"] = pd.to_datetime(ehs_demand["Date"])

ehs_demand.dtypes

Zone              object
Date      datetime64[ns]
Actual           float64
dtype: object

## Aggregate monthly EMS demand by zone

EMS demand is defined as the total number of Emergency Health Services responses per month aggregated by Nova Scotia Health Zone.

In [7]:
ehs_monthly = (
    ehs_demand
    .groupby(["Zone", "Date"], as_index=False)["Actual"]
    .sum()
    .rename(columns={"Actual": "ehs_responses"})
)

ehs_monthly.head()

,Zone,Date,ehs_responses
0,Central,2021-03-01,344.0
1,Central,2021-04-01,3553.0
2,Central,2021-05-01,4805.0
3,Central,2021-06-01,3737.0
4,Central,2021-07-01,3766.0


## Remove unknown zone records

Exclude records without a valid Nova Scotia Health zone assignment to maintain geographic consistency across system signals.

In [8]:
ehs_monthly = ehs_monthly[ehs_monthly["Zone"] != "Unknown"].copy()

ehs_monthly["Zone"].unique()

array(['Central', 'Eastern', 'Northern', 'Western'], dtype=object)

## Engineer EMS demand trend features

Create a 3-month rolling mean and month-to-month percent change to capture short-term EMS demand trends.

In [9]:
ehs_monthly = ehs_monthly.sort_values(["Zone", "Date"])

ehs_monthly["responses_roll3"] = (
    ehs_monthly
    .groupby("Zone")["ehs_responses"]
    .transform(lambda x: x.rolling(3, min_periods=1).mean())
)

ehs_monthly["responses_pct_change"] = (
    ehs_monthly
    .groupby("Zone")["ehs_responses"]
    .pct_change()
)

ehs_monthly.head()

,Zone,Date,ehs_responses,responses_roll3,responses_pct_change
0,Central,2021-03-01,344.0,344.000000,NaN
1,Central,2021-04-01,3553.0,1948.500000,9.328488
2,Central,2021-05-01,4805.0,2900.666667,0.352378
3,Central,2021-06-01,3737.0,4031.666667,-0.222268
4,Central,2021-07-01,3766.0,4102.666667,0.007760


## Validate EMS demand distribution by zone

Summary statistics are reviewed to confirm that aggregated EMS response counts fall within plausible ranges for each Nova Scotia Health zone and to identify potential anomalies or partial reporting periods.

In [10]:
ehs_monthly.groupby("Zone")["ehs_responses"].describe()

,count,mean,std,min,25%,50%,75%,max
Zone,,,,,,,,
Central,34.0,3904.529412,792.321110,344.0,3641.75,3781.0,4469.75,5082.0
Eastern,34.0,1821.235294,380.828925,154.0,1664.50,1802.5,2037.50,2431.0
Northern,34.0,1997.470588,418.083233,161.0,1843.50,1973.0,2283.75,2562.0
Western,34.0,2349.205882,472.519987,231.0,2161.25,2300.5,2722.25,3095.0


## Assumptions and limitations

- EMS demand is represented by monthly EHS response counts aggregated by zone.
- Early months may contain partial or lower counts due to dataset start or reporting ramp-up.
- EMS responses represent service events, not unique patients.
- Demand increases may reflect population, utilization, or reporting changes rather than unmet need alone.

## Save processed EMS demand signal

Export the cleaned and aggregated EMS demand time series for integration with other health system signals.

In [11]:
ehs_monthly.to_csv(
    "data_processed/ehs_demand_ns.csv",
    index=False
)